# MAVE-Q v4 (`VetQuant-RSI`) — 100% Free Cloud Training Notebook

> **Supported $0 Cloud Tiers**: Google Colab Free (`1x T4 16GB`), Kaggle Free (`2x T4 32GB`, 30h/wk), Kaggle Free (`TPU v5e-8 128GB HBM`, 20h/wk), Lightning AI / Modal Free Tier.  
> **Peak VRAM Budget**: `< 4.5 GB VRAM` for complete MAVE-Q v4 (`kaggle_t4_optimal`, ~10M structural core) + `Qwen3-1.7B/4B` 4-bit QLoRA SFT + DPO (with automatic inter-stage VRAM cleanup & `--resume` checkpoint recovery for 12h free sessions).  
> **Disclaimer**: Strictly academic research & simulation ONLY. Zero live trading, zero brokerage APIs, zero financial advice.

In [ ]:
# Cell 1: Detect Free Cloud Accelerator (Colab T4 / Kaggle T4 x2 / Kaggle TPU v5e-8 / Lightning / Local)
import os
platform = 'Kaggle' if os.path.exists('/kaggle/working') else ('Colab' if os.path.exists('/content') else 'Local')
print('Detected Platform:', platform)
try:
    import torch
    print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available(), '| GPUs:', torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {p.name} ({p.total_memory / 1024**3:.2f} GB VRAM, bf16={torch.cuda.is_bf16_supported()})')
except Exception as e:
    print('PyTorch check:', e)
if os.environ.get('TPU_NAME') or os.environ.get('PJRT_DEVICE') == 'TPU':
    print('TPU v5e-8 detected! Using torch_xla BF16 path.')

In [ ]:
# Cell 2: Clone / Pull MAVE-Q v4 Repository
import os
REPO_URL = 'https://github.com/KenyaWashed/VN_Trading_Agents.git'
REPO_DIR = 'VN_Trading_Agents'
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git pull --ff-only || true

In [ ]:
# Cell 3: Safe Dependency Install (Preserves Preinstalled Kaggle/Colab CUDA PyTorch!)
!pip install -q pydantic pyyaml pyarrow scipy scikit-learn httpx pytest ruff mypy
!pip install -q --no-deps -e .
# Optional 4-bit QLoRA + DPO deps (< 4.5 GB VRAM on Free T4)
!pip install -q peft bitsandbytes trl datasets accelerate transformers || true

In [ ]:
# Cell 4: Run Mandatory Anti-Leakage, Shift-Invariance, & Seal Checks BEFORE Training
!pytest tests/test_sandbox_seal.py tests/test_maveq_v4.py tests/test_veteran_reasoning.py -q

In [ ]:
# Cell 5: Free-Tier Training Configuration (Edit Here)
import os
PRESET = 'auto'                # auto -> kaggle_t4_optimal (Colab 1xT4) | kaggle_max_100m (Kaggle 2xT4) | tpu_v5e_350m (TPU)
MODE = 'maveq_v4'              # 'maveq_v4' | 'full'
SLM_MODEL = 'Qwen/Qwen3-1.7B'  # Fits < 3.5 GB VRAM in 4-bit QLoRA+DPO (use 'Qwen/Qwen3-4B' for < 4.5 GB VRAM)
ENABLE_VETERAN_DPO = True      # 2-Stage 4-bit QLoRA SFT -> DPO on 10-Year Veteran reasoning trajectories
EPOCHS = 10
ROUNDS = 6
CKPT_DIR = '/kaggle/working/maveq_v4_ckpts' if os.path.exists('/kaggle/working') else 'artifacts/checkpoints'
print('Checkpoints & --resume state dir:', CKPT_DIR)

In [ ]:
# Cell 6: Train Full MAVE-Q v4 + 10-Year Veteran QLoRA/DPO with Automatic --resume & VRAM Cleanup
dpo_flag = '--enable-veteran-dpo' if ENABLE_VETERAN_DPO else ''
!python train_colab_kaggle.py \
    --preset {PRESET} \
    --mode {MODE} \
    --slm-model {SLM_MODEL} \
    {dpo_flag} \
    --epochs {EPOCHS} \
    --rounds {ROUNDS} \
    --checkpoint-dir {CKPT_DIR} \
    --resume

In [ ]:
# Cell 7: Ask the Trained 10-Year Veteran Agent Natural-Language Market Questions!
!python scripts/ask_veteran_agent.py --symbol FPT --checkpoint-dir {CKPT_DIR} --question "Đánh giá cơ hội mua FPT và rủi ro T+2.5 hiện tại?"
!python scripts/ask_veteran_agent.py --symbol NVL_STRESS --checkpoint-dir {CKPT_DIR} --question "Có nên bắt đáy NVL_STRESS khi cổ phiếu giảm mạnh không?"

In [ ]:
# Cell 8: Run Veteran 32-Scenario Benchmark + Multi-Seed Ablations & OOD Evaluation
!python experiments/run_veteran_reasoning_eval.py
!python experiments/run_v4_ablation_and_ood.py --seeds '11,22,33,44,55,66,77,2026' --workers 4